# ZIPコード単位 Areal Deprivation Index（ADI）作成手順

## 1. 概要

本手順書は、2020年国勢調査の小地域集計を用いて **全国の小地域レベルADI** を作成し、それを **郵便番号ポリゴンへ面積按分して全国の郵便番号レベルADI** を作成するための手順である。  
今回の改訂では、従来版で問題になっていた以下の点を修正した。

- `"-"` と `"X"` の扱いを明確化
- 秘匿地域・合算地域の処理を `final_unit_code` ベースに統一
- `keep_finest` による通常階層の重複除去を導入
- 統計側の町丁字コードとポリゴン側のコードの不一致を修正
- 小地域版と郵便番号版を別 notebook に分離
- 出力先とファイル命名を整理

本プロセスは、以下の **2本の notebook** に分けて実行する。

1. **Notebook 1：小地域レベルADI（地理情報付き）作成**
2. **Notebook 2：郵便番号レベルADI（地理情報付き）作成**

---

## 2. Notebook名

- **Notebook 1**：`ArealDeprivationIndex_smallarea.ipynb`
- **Notebook 2**：`ArealDeprivationIndex_ziparea.ipynb`

## 3. 命名ルール

- `full`：欠損や非居住も含む完全版
- `analysis`：分析用に絞った版
- `attribute`：geometryなしの属性表
- `min`：必要最小限列の軽量版

---

## 4. 使用データ

### 3.1 国勢調査（2020年）小地域集計

使用表：

| 表番号 | 内容 |
|---|---|
| 第6表 | 世帯構成 |
| 第7表 | 住宅の所有形態 |
| 第9表 | 労働力状態 |
| 第12表 | 職業 |

### 3.2 小地域ポリゴン

- 国勢調査2020 小地域ポリゴン
- 保存場所：`Processed/Geospatial/Town_polygon/2020/smallarea_2020_alljapan.gpkg`

### 3.3 郵便番号ポリゴン

- GEOSPACE 郵便番号ポリゴン（2022）
- 保存場所：`Raw/Geospatial/ZIP_polygon/2022/.../shape/GSP99.shp`

---

## 5. 国勢調査データの基本ルール

### 4.1 記号の扱い

- `"-"`：**0として扱う**
- `"X"`：**欠損（秘匿）として扱う**
- `"…"`：欠損として扱う

### 4.2 行の絞り込み

- `男女 == "総数"` を使用
- 年齢・世帯分類のある表では `総数` のみ使用

---

## 6. 小地域レベルADI作成の考え方（Notebook 1）

## 6.1 重要なキー：`final_unit_code`

今回の改訂版では、統計表上の最終観測単位を `final_unit_code` として定義する。  
これは、通常の階層処理と秘匿・合算処理の両方を反映したコードである。

### 5.1.1 秘匿・合算の扱い

- `秘匿地域`：`秘匿先情報` に統合
- `合算地域あり`：その行自身を代表単位とし、`合算地域` 列の構成コードを同一グループに結ぶ
- `Union-Find` でグループ化し、同じ公開単位に属するコードへ統一

### 5.1.2 通常階層の重複除去

`keep_finest` を作り、以下のルールで採用単位を決める。

- より細かい下位コードが存在する粗い単位は落とす
- ただし `合算地域あり` の行は公開単位なので残す
- これにより、たとえば
  - `平河町`（lv3）と `平河町一丁目・二丁目`（lv4）が両方ある場合は lv4 を採用
  - `松が谷` のように下位が無い単独 lv2 地域は lv2 をそのまま採用

---

## 6.2 コード正規化の改訂点

### 5.2.1 統計側の `町丁字コード`

統計側では、**桁不足の場合は右側を 0 埋め** して shapefile 側の階層表現に合わせる。

例：

- `600` → `600000` ではなく、元の階層解釈に応じて **右0埋めで6桁化**
- `0110` → `011000`
- `0011` → `001100`


### 5.2.2 ポリゴン側のコード

小地域ポリゴンでは、`S_AREA` ではなく **`KEY_CODE` を優先** して解釈する。  
ただし、`KEY_CODE` が桁不足の場合は **右側に0を付与して11桁化** する。

例：

- `131040011` → `13104001100`
- 先頭5桁 `13104` が市区町村コード
- 後ろ6桁 `001100` が町丁字コード

この修正により、従来版で発生していた

- `13104_001101` vs `13104_001100`
- `13101_011001` vs `13101_011000`

のようなズレを解消した。

---

## 6.3 小地域ADIの構成変数

以下の8変数を使用する。

| 変数名 | 定義 |
|---|---|
| `old_couple_rate` | 高齢夫婦のみ世帯 / 全世帯 |
| `old_single_rate` | 高齢単身世帯 / 全世帯 |
| `single_parent_proxy_rate` | ひとり親世帯 proxy / 全世帯 |
| `rent_rate` | 借家世帯 / 住宅に住む一般世帯 |
| `sales_service_rate` | 販売・サービス・保安職 / 職業総数 |
| `agriculture_rate` | 農林漁業従事者 / 職業総数 |
| `blue_collar_rate` | 生産工程・輸送・建設・運搬等 / 職業総数 |
| `unemployment_rate` | （労働力人口 − 職業総数）/ 労働力人口 |

### ADI式

```python
ADI_raw = (
    2.99 * old_couple_rate
    + 7.57 * old_single_rate
    + 17.4 * single_parent_proxy_rate
    + 2.22 * rent_rate
    + 4.03 * sales_service_rate
    + 6.05 * agriculture_rate
    + 5.38 * blue_collar_rate
    + 18.3 * unemployment_rate
)
```

---

## 6.4 非居住地域の扱い

小地域レベルでは以下のフラグを作る。

```python
exclude_nonresidential = (
    (総数 == 0) |
    (住宅に住む一般世帯 == 0)
)
```

この結果、

- `exclude_nonresidential == False` かつ `ADI_raw.notna()`  
  → **analysis版**
- それ以外も含める  
  → **full版**

とする。

---

## 6.5 Notebook 1 の出力

### 統計属性表
- `smallarea_adi_attribute_final_unit.parquet`

### 地理情報付き full版
- `smallarea_adi_full.gpkg`
- `smallarea_adi_full.parquet`

### 地理情報付き analysis版
- `smallarea_adi_analysis.gpkg`
- `smallarea_adi_analysis.parquet`

### 軽量版
- `smallarea_adi_full_min.gpkg`
- `smallarea_adi_full_min.parquet`

---

## 7. 郵便番号レベルADI作成の考え方（Notebook 2）

## 7.1 入力

Notebook 2 では、Notebook 1 で作成した **`smallarea_adi_analysis.gpkg`** を入力とする。  
つまり、郵便番号版は **分析対象小地域のみ** を面積按分して作る。

---

## 7.2 面積按分

`smallarea × zip` の intersection をとり、各小地域の ADI を郵便番号へ配分する。

### 重み

```python
area_weight = intersect_area / smallarea_area
```

### 加重和

```python
ADI_weighted = ADI_raw * area_weight
```

### 郵便番号単位集約

```python
zip_adi = Σ(ADI_weighted) / Σ(area_weight)
```

ここでの `Σ(area_weight)` は **郵便番号内での有効寄与面積の合計** であり、  
`final_key` 単位にみると、理想的には `area_weight_sum ≈ 1` になる。

---

## 7.3 重みの品質確認

`final_key` ごとに

```python
area_weight_sum = Σ(intersect_area / smallarea_area)
```

を確認する。

期待される結果：

- 多くの地域で `area_weight_sum = 1`
- `25%`, `50%`, `75%`, `max` が 1 でも問題ない
- `min` が極端に小さい場合は、海岸線・境界ズレ・特殊地域の可能性を確認

---

## 7.4 郵便番号ポリゴンの dissolve

同一郵便番号に複数ポリゴンがあるため、保存前に **郵便番号単位で dissolve** する。

```python
zip_poly = zip_use[[zip_col, "geometry"]].dissolve(by=zip_col, as_index=False)
```

---

## 7.5 郵便番号版の full / analysis

### full版
- 郵便番号ポリゴン全体に left join
- ADIが無い郵便番号も含む

### analysis版
- `ADI_raw.notna()` の郵便番号のみ残す

---

## 7.6 郵便番号版の出力

### full版
- `zip_adi_full.csv`
- `zip_adi_full.gpkg`
- `zip_adi_full.parquet`

### analysis版
- `zip_adi_analysis.csv`
- `zip_adi_analysis.gpkg`
- `zip_adi_analysis.parquet`

---

## 8. マッピング上の注意

### 7.1 東京都の切り出し

ZIP版の成果物には都道府県名列を持たない場合があるため、東京都表示時は  
**元の郵便番号ポリゴンから `市区町村CD` 先頭2桁 = `"13"` で東京都を切り出し、ADIを再結合** する。

### 7.2 `PREF_NAME` が無いのは正常

`zip_adi_analysis.gpkg` に都道府県名列が無いのは、保存時に列を絞ったためである。  
必要に応じて元ポリゴンから再付与する。

---

## 9. 従来版からの主な変更点

| 項目 | 従来版 | 改訂版 |
|---|---|---|
| `-` の扱い | 欠損寄り | **0** |
| `X` の扱い | 明確でない | **欠損** |
| 秘匿地域 | 単純除外寄り | **`final_unit_code` に統合** |
| 階層処理 | fallback中心 | **`keep_finest` 導入** |
| ポリゴン側キー | `S_AREA` ベース | **`KEY_CODE` ベース + 右0埋め** |
| 小地域出力 | 混在 | **smallareaフォルダに整理** |
| ZIP出力 | 混在 | **zipareaフォルダに整理** |

---

## 10. 解釈上の注意

### 9.1 ADI欠損の主因

小地域版の欠損は主に以下で生じる。

- 非居住地域
- 総数や住宅世帯数が0の地域
- 秘匿・合算の影響で単体値が定義されない地域

### 9.2 郵便番号版の欠損

ZIP版の欠損は主に以下で生じる。

- 分析対象小地域と交差しない郵便番号
- 非居住小地域しか含まない郵便番号
- 特殊郵便番号領域

---

## 11. 最終成果物

### 小地域版
- 分析用：`smallarea_adi_analysis.parquet`
- GIS確認用：`smallarea_adi_analysis.gpkg`
- 完全版保管用：`smallarea_adi_full.parquet / gpkg`

### 郵便番号版
- 分析用：`zip_adi_analysis.parquet`
- GIS確認用：`zip_adi_analysis.gpkg`
- 完全版保管用：`zip_adi_full.parquet / gpkg`

---

## 12. 再現手順（改訂版）

### Notebook 1
1. 4表を読み込み
2. `"-"` を0、`"X"` を欠損として処理
3. `final_unit_code` を作成
4. `keep_finest` を作成
5. `keep_finest=True` の行だけ残す
6. `final_unit_code` 単位で集約
7. 4表を merge
8. ADIを計算
9. ポリゴン側に `final_key` を付与
10. `dissolve` して小地域ADI地理データを保存

### Notebook 2
1. `smallarea_adi_analysis.gpkg` を読む
2. 郵便番号ポリゴンを読む
3. intersection
4. 面積重み計算
5. 郵便番号単位に集約
6. 郵便番号ポリゴンを dissolve
7. full版とanalysis版を保存

---

## 13. まとめ

本改訂版では、従来の手順に対して

- 秘匿・合算処理
- 階層処理
- ポリゴンキーの整合
- 出力管理

を大幅に改善した。

特に重要なのは、

- **統計側とポリゴン側で同じ `final_key` を持つこと**
- **analysis版とfull版を分けて保存すること**
- **ZIP版は小地域版 analysis から作ること**

である。

この改訂により、統計側とポリゴン側の key 不一致や歯抜け問題を解消できる。